# Crear Job
Automate job creation and management
https://docs.databricks.com/aws/en/jobs/automate

In [0]:
%pip install --upgrade databricks-sdk
%restart_python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 892.1/892.1 kB 27.4 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-e9e64f6c-afa9-4377-84bb-0582868e903f
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.67.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-e9e64f6c-afa9-4377-84bb-0582868e903f
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.65.0 requires protobuf!=3.20.0,

In [0]:
from databricks.sdk.service.jobs import JobSettings as Job
from databricks.sdk import WorkspaceClient

job_name = "WF_lectura_base" 

job_body = Job.from_dict(
{
  "name": job_name,
  "email_notifications": {
    "no_alert_for_skipped_runs": False
  },
  "webhook_notifications": {},
  "timeout_seconds": 0,
  "max_concurrent_runs": 1,
  "tasks": [
    {
      "task_key": "PL_cargar_archivo_base",
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/Utilidades/Validaciones/NB_lectura_archivo_base",
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Pl_lectura_carpetas_catalogo",
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/Utilidades/Validaciones/NB_lectura_carpetas",
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "validacion_archivo_base",
      "depends_on": [
        {
          "task_key": "PL_cargar_archivo_base"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "condition_task": {
        "op": "EQUAL_TO",
        "left": "{{tasks.PL_cargar_archivo_base.values.estado}}",
        "right": "true"
      },
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "PL_validar_archivo_destino",
      "depends_on": [
        {
          "task_key": "validacion_archivo_base",
          "outcome": "true"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/Utilidades/Validaciones/NB_validar_hoja_archivo_destino",
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Completitud_archivo_destino",
      "depends_on": [
        {
          "task_key": "PL_validar_archivo_destino"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "condition_task": {
        "op": "EQUAL_TO",
        "left": "{{tasks.PL_validar_archivo_destino.values.estado}}",
        "right": "true"
      },
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "NB_notificaciones_fallo_archivo_destino",
      "depends_on": [
        {
          "task_key": "Completitud_archivo_destino",
          "outcome": "false"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/Utilidades/NB_notificaciones_logicapp",
        "base_parameters": {
          "resultado": "{{tasks.PL_validar_archivo_destino.values.resultado}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "NB_notificaciones_fallo_archivo_base",
      "depends_on": [
        {
          "task_key": "validacion_archivo_base",
          "outcome": "false"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/Utilidades/NB_notificaciones_logicapp",
        "base_parameters": {
          "resultado": "{{tasks.PL_cargar_archivo_base.values.resultado}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "validacion_carpetas_catalogo",
      "depends_on": [
        {
          "task_key": "Pl_lectura_carpetas_catalogo"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "condition_task": {
        "op": "EQUAL_TO",
        "left": "{{tasks.Pl_lectura_carpetas_catalogo.values.estado}}",
        "right": "true"
      },
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "PL_validacion_carpetas",
      "depends_on": [
        {
          "task_key": "validacion_carpetas_catalogo",
          "outcome": "true"
        },
        {
          "task_key": "Completitud_archivo_destino",
          "outcome": "true"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/Utilidades/Validaciones/NB_validacion_carpeta_datos",
        "base_parameters": {
          "df_excel_base": "{{tasks.PL_cargar_archivo_base.values.df_excel_base}}",
          "df_carpetas_catalogo": "{{tasks.Pl_lectura_carpetas_catalogo.values.df_carpetas_catalogo}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Completitud_Carpetas",
      "depends_on": [
        {
          "task_key": "PL_validacion_carpetas"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "condition_task": {
        "op": "EQUAL_TO",
        "left": "{{tasks.PL_validacion_carpetas.values.estado}}",
        "right": "true"
      },
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "PL_validacion_archivos",
      "depends_on": [
        {
          "task_key": "Completitud_Carpetas",
          "outcome": "true"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/Utilidades/Validaciones/NB_validacion_archivos_carpetas",
        "base_parameters": {
          "df_base": "{{tasks.PL_validacion_carpetas.values.df_base}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Completitud_Archivos",
      "depends_on": [
        {
          "task_key": "PL_validacion_archivos"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "condition_task": {
        "op": "EQUAL_TO",
        "left": "{{tasks.PL_validacion_archivos.values.estado}}",
        "right": "true"
      },
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "PL_validar_hojas",
      "depends_on": [
        {
          "task_key": "Completitud_Archivos",
          "outcome": "true"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/Utilidades/Validaciones/NB_validar_hoja_archivo",
        "base_parameters": {
          "df_listado_archivos": "{{tasks.PL_validacion_archivos.values.df_listado_archivos}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Completitud_Hojas",
      "depends_on": [
        {
          "task_key": "PL_validar_hojas"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "condition_task": {
        "op": "EQUAL_TO",
        "left": "{{tasks.PL_validar_hojas.values.estado}}",
        "right": "true"
      },
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Generador_Delta_Table_EEFF",
      "depends_on": [
        {
          "task_key": "Completitud_Hojas",
          "outcome": "true"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/GeneradorDeltaTables/NB_Lectura_EEFF_TE_General",
        "base_parameters": {
          "df_base": "{{tasks.PL_validar_hojas.values.df_listado_archivos}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Generador_Delta_Table_bonos_creditos",
      "depends_on": [
        {
          "task_key": "Generador_Delta_Table_EEFF"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/GeneradorDeltaTables/NB_Lectura_deuda_consolidada_bonos_creditos",
        "base_parameters": {
          "df_base": "{{tasks.PL_validar_hojas.values.df_listado_archivos}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Generador_Delta_Table_CAPEX",
      "depends_on": [
        {
          "task_key": "Generador_Delta_Table_bonos_creditos"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/GeneradorDeltaTables/NB_Lectura_CAPEX",
        "base_parameters": {
          "df_base": "{{tasks.PL_validar_hojas.values.df_listado_archivos}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Generador_Delta_Table_participacion_empresas",
      "depends_on": [
        {
          "task_key": "Generador_Delta_Table_CAPEX"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/GeneradorDeltaTables/NB_Lectura_participacion_compañias",
        "base_parameters": {
          "df_base": "{{tasks.PL_validar_hojas.values.df_listado_archivos}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Generador_Delta_Table_perfil_deuda",
      "depends_on": [
        {
          "task_key": "Generador_Delta_Table_participacion_empresas"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/GeneradorDeltaTables/NB_Lectura_perfil_deuda",
        "base_parameters": {
          "df_base": "{{tasks.PL_validar_hojas.values.df_listado_archivos}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Generador_Delta_Table_proyectos",
      "depends_on": [
        {
          "task_key": "Generador_Delta_Table_perfil_deuda"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/GeneradorDeltaTables/NB_lectura_proyectos",
        "base_parameters": {
          "df_base": "{{tasks.PL_validar_hojas.values.df_listado_archivos}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Generador_Delta_Table_trafico_vias",
      "depends_on": [
        {
          "task_key": "Generador_Delta_Table_proyectos"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/GeneradorDeltaTables/NB_Lectura_trafico_vias",
        "base_parameters": {
          "df_base": "{{tasks.PL_validar_hojas.values.df_listado_archivos}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Generador_kit_EEFF",
      "depends_on": [
        {
          "task_key": "Completitud_Hojas",
          "outcome": "true"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/GeneradorKit/Generador_Kit_EEFF_TE",
        "base_parameters": {
          "df_base": "{{tasks.PL_validar_hojas.values.df_listado_archivos}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Generador_kit_bonos_creditos",
      "depends_on": [
        {
          "task_key": "Generador_kit_EEFF"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/GeneradorKit/Generador_Kit_deudas_bonos_creditos",
        "base_parameters": {
          "df_base": "{{tasks.PL_validar_hojas.values.df_listado_archivos}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Generador_CAPEX",
      "depends_on": [
        {
          "task_key": "Generador_kit_bonos_creditos"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/GeneradorKit/Generador_Kit_Capex",
        "base_parameters": {
          "df_base": "{{tasks.PL_validar_hojas.values.df_listado_archivos}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Generador_participacion_companias",
      "depends_on": [
        {
          "task_key": "Generador_CAPEX"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/GeneradorKit/Generador_Kit_participacion_compañias",
        "base_parameters": {
          "df_base": "{{tasks.PL_validar_hojas.values.df_listado_archivos}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Generador_perfil_deuda",
      "depends_on": [
        {
          "task_key": "Generador_participacion_companias"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/GeneradorKit/Generador_Kit_perfil_deuda",
        "base_parameters": {
          "df_base": "{{tasks.PL_validar_hojas.values.df_listado_archivos}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Generador_proyectos",
      "depends_on": [
        {
          "task_key": "Generador_perfil_deuda"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/GeneradorKit/Generador_Kit_proyectos",
        "base_parameters": {
          "df_base": "{{tasks.PL_validar_hojas.values.df_listado_archivos}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Generador_trafico_vias",
      "depends_on": [
        {
          "task_key": "Generador_proyectos"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/GeneradorKit/Generador_Kit_trafico_vias",
        "base_parameters": {
          "df_base": "{{tasks.PL_validar_hojas.values.df_listado_archivos}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "Nb_notifiaciones_fallo_completitud_hojas",
      "depends_on": [
        {
          "task_key": "Completitud_Hojas",
          "outcome": "false"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/Utilidades/NB_notificaciones_logicapp",
        "base_parameters": {
          "resultado": "{{tasks.PL_validar_hojas.values.resultado}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "NB_notificaciones_fallo_completitud_archivos",
      "depends_on": [
        {
          "task_key": "Completitud_Archivos",
          "outcome": "false"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/Utilidades/NB_notificaciones_logicapp",
        "base_parameters": {
          "resultado": "{{tasks.PL_validacion_archivos.values.resultado}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "NB_notificaciones_fallo_carpeta_catalogo",
      "depends_on": [
        {
          "task_key": "validacion_carpetas_catalogo",
          "outcome": "false"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/Utilidades/NB_notificaciones_logicapp",
        "base_parameters": {
          "resultado": "{{tasks.Pl_lectura_carpetas_catalogo.values.resultado}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    },
    {
      "task_key": "validacion_completitud_carpetas",
      "depends_on": [
        {
          "task_key": "Completitud_Carpetas",
          "outcome": "false"
        }
      ],
      "run_if": "ALL_SUCCESS",
      "notebook_task": {
        "notebook_path": "/Workspace/Proyectos_Dev/databricks-medallion/KitValoracion/Notebooks/Utilidades/NB_notificaciones_logicapp",
        "base_parameters": {
          "resultado": "{{tasks.PL_validacion_carpetas.values.resultado}}"
        },
        "source": "WORKSPACE"
      },
      
      "timeout_seconds": 0,
      "email_notifications": {},
      "webhook_notifications": {}
    }
  ],
  "git_source": {
    "git_url": "https://dev.azure.com/ISA-InterconexionElectrica/ISA_Datalake_Plantillas_Ingesta_MVM/_git/ISA_Datalake_Plantillas_Ingesta_MVM",
    "git_provider": "azureDevOpsServices",
    "git_branch": "develop"
  },
  "queue": {
    "enabled": True
  },
  "parameters": [
    {
      "name": "Anio",
      "default": "2024"
    },
    {
      "name": "CatalogoDelta",
      "default": "kitvaloracion"
    },
    {
      "name": "EsquemaDelta",
      "default": "base"
    },
    {
      "name": "ruta_archivo_destino",
      "default": "/Volumes/kitvaloracion/base/KitValoracion/2024/3Q24/Kit consolidado y separados/kit_Inversionistas ISA_Sp_3Q24.xlsx"
    },
    {
      "name": "ruta_catalogo",
      "default": "/Volumes/kitvaloracion/base/KitValoracion/2024/3Q24/"
    },
    {
      "name": "ruta_excel_base",
      "default": "/Volumes/kitvaloracion/base/config/NombreCarpetas.xlsx"
    },
    {
      "name": "ruta_kit_origen",
      "default": "/Volumes/kitvaloracion/base/KitValoracion/2024/3Q24/Kit consolidado y separados/kit_Inversionistas ISA_Sp.xlsx"
    },
    {
      "name": "Trimestre",
      "default": "3"
    }
  ],
  "performance_target": "STANDARD"
}
)

w = WorkspaceClient()

# Buscar si el job ya existe por nombre
existing_jobs = [job for job in w.jobs.list() if job.settings and job.settings.name == job_name]

if existing_jobs:
    job_id = existing_jobs[0].job_id
    # w.jobs.update(job_id=job_id, **job_body.as_shallow_dict())
    print(f"Job existe debe borrarlo antes de crearlo de nuevo. Ver el job en {w.config.host}/#job/{job_id}\n")
else:
    j = w.jobs.create(**job_body.as_shallow_dict())
    print(f"Job creado. Ver el job en {w.config.host}/#job/{j.job_id}\n")

Job existe debe borrarlo antes de crearlo de nuevo. Ver el job en https://dbc-c4a7b866-5204.cloud.databricks.com/#job/217796283891466

